In [0]:
# ============================================================
# TRANSFORMATION NOTEBOOK — setup
# ============================================================
from pyspark.sql import functions as F

# Load the three cleaned silver tables (the hand-off from the cleaning notebook)
df_derm = spark.table("dermstore.dermstoreschema.silver_dermstore")
df_seph = spark.table("dermstore.dermstoreschema.silver_sephora")
df_cos  = spark.table("dermstore.dermstoreschema.silver_cosing")

# Quick confirm they loaded
print("Dermstore rows:", df_derm.count())
print("Sephora rows:  ", df_seph.count())
print("CosIng rows:   ", df_cos.count())

Dermstore rows: 120
Sephora rows:   8495
CosIng rows:    31759


In [0]:
# ============================================================
# TRANSFORMATION — Step 1: Explode Dermstore ingredients into a bridge table
# ============================================================
# Turn "one product with a list of ingredients" into
# "one row per product-ingredient pair" (a bridge table).
# This makes "which products contain X?" easy, and sets up
# the CosIng validation join in the next step.

# F.explode takes each item in ingredients_list and gives it its own row.
derm_bridge = df_derm.select(
    "product_id",
    "brand",
    F.explode("ingredients_list").alias("ingredient")  # one row per ingredient
)

# Clean the exploded ingredient: trim spaces + uppercase, so it will
# match CosIng's inci_name (which we already uppercased) in the next step.
derm_bridge = derm_bridge.withColumn(
    "ingredient_clean",
    F.upper(F.trim(F.col("ingredient")))
)

print("Dermstore products:", df_derm.count())
print("Bridge rows (product-ingredient pairs):", derm_bridge.count())
derm_bridge.select("product_id", "ingredient_clean").show(15, truncate=50)

Dermstore products: 120
Bridge rows (product-ingredient pairs): 4245
+----------+----------------------------------+
|product_id|                  ingredient_clean|
+----------+----------------------------------+
|  13441438|                      AQUA (WATER)|
|  13441438|                ACETYL GLUCOSAMINE|
|  13441438|                CYCLOPENTASILOXANE|
|  13441438|                  TRIETHYL CITRATE|
|  13441438|                  CETEARYL ALCOHOL|
|  13441438|              C1215 ALKYL BENZOATE|
|  13441438|BUTYROSPERMUM PARKII (SHEA) BUTTER|
|  13441438|                 GLYCERYL STEARATE|
|  13441438|                          GLYCERIN|
|  13441438|        ISOCETYL STEAROYL STEARATE|
|  13441438|             ETHOXYDIGLYCOL OLEATE|
|  13441438|                       DIMETHICONE|
|  13441438|                 CYCLOHEXASILOXANE|
|  13441438|                    ETHOXYDIGLYCOL|
|  13441438|                   PEG100 STEARATE|
+----------+----------------------------------+
only showing top 15

In [0]:
# ============================================================
# TRANSFORMATION — Step 2: Validate ingredients against CosIng
# ============================================================
# Join the bridge table to the CosIng registry on the ingredient name.
# Matched ingredients get CosIng's "function" info; unmatched ones get null.
# A "left" join keeps ALL bridge rows (so we can see what didn't match too).

derm_validated = derm_bridge.join(
    df_cos,                                              # the CosIng registry
    derm_bridge.ingredient_clean == df_cos.inci_name,    # match condition
    how="left"                                           # keep all bridge rows
)

# How many ingredient rows matched a real CosIng entry vs not?
total = derm_validated.count()
matched = derm_validated.filter(F.col("inci_name").isNotNull()).count()
print(f"Total ingredient rows: {total}")
print(f"Matched in CosIng:     {matched}  ({matched/total*100:.1f}%)")
print(f"Not matched:           {total - matched}")

Total ingredient rows: 4255
Matched in CosIng:     2909  (68.4%)
Not matched:           1346


In [0]:
# Look at the ingredients that did NOT match CosIng, most common first.
# This is our data-quality report: WHY didn't they match?
derm_validated.filter(F.col("inci_name").isNull()) \
    .groupBy("ingredient_clean") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(30, truncate=60)

+------------------------------------------------------------+-----+
|                                            ingredient_clean|count|
+------------------------------------------------------------+-----+
|                                  FOR THE LATEST INFORMATION|  109|
|IT IS RECOMMENDED TO REVIEW THE INGREDIENT LIST PRINTED O...|  109|
|                                                           1|   28|
|                                                2-HEXANEDIOL|   26|
|                          BUTYROSPERMUM PARKII (SHEA) BUTTER|   23|
|                                              AQUA/WATER/EAU|   20|
|                      HELIANTHUS ANNUUS (SUNFLOWER) SEED OIL|   17|
|                                          FRAGRANCE (PARFUM)|   13|
|                                              WATER/AQUA/EAU|   13|
|                                            PARFUM/FRAGRANCE|   10|
|                       LAVANDULA ANGUSTIFOLIA (LAVENDER) OIL|   10|
|                      SIMMONDSIA 

In [0]:
# ============================================================
# TRANSFORMATION — Step 2b: Targeted ingredient normalization
# ============================================================
# From the unmatched report, we fix the highest-value problems so
# more ingredients match CosIng. Each rule targets a category we saw.

# Start from the cleaned ingredient we already made
norm = derm_bridge

# --- Rule 1: drop scraped JUNK text (not real ingredients) ---
# These boilerplate phrases got scraped into the ingredient field.
junk_phrases = ["FOR THE LATEST INFORMATION", "IT IS RECOMMENDED TO REVIEW"]
junk_cond = F.lit(False)
for p in junk_phrases:
    junk_cond = junk_cond | F.col("ingredient_clean").contains(p)
# also drop pure-number / 1-char junk like "1"
junk_cond = junk_cond | (F.length(F.trim(F.col("ingredient_clean"))) <= 1)
norm = norm.filter(~junk_cond)   # ~ means NOT — keep rows that are NOT junk

# --- Rule 2: strip color-index codes like "(CI 77891)" ---
# regexp_replace removes the "(CI #####)" part wherever it appears.
norm = norm.withColumn(
    "ingredient_norm",
    F.regexp_replace(F.col("ingredient_clean"), r"\s*\(?CI\s*\d+\)?", "")
)

# --- Rule 3: normalize all the WATER variants to just "AQUA" ---
# AQUA/WATER/EAU, WATER (AQUA), AQUA (WATER), etc. -> "AQUA"
norm = norm.withColumn(
    "ingredient_norm",
    F.when(F.col("ingredient_norm").rlike(r"^(AQUA|WATER|EAU)([\s/(].*)?$"), "AQUA")
     .otherwise(F.col("ingredient_norm"))
)

# --- Rule 4: remove trailing punctuation and extra spaces ---
# strips a trailing "." and trims, e.g. "LIMONENE." -> "LIMONENE"
norm = norm.withColumn(
    "ingredient_norm",
    F.trim(F.regexp_replace(F.col("ingredient_norm"), r"[.\s]+$", ""))
)

# Re-join to CosIng using the NEW normalized column
derm_validated = norm.join(
    df_cos,
    norm.ingredient_norm == df_cos.inci_name,
    how="left"
)

# Measure the new match rate
total = derm_validated.count()
matched = derm_validated.filter(F.col("inci_name").isNotNull()).count()
print(f"Total ingredient rows: {total}")
print(f"Matched in CosIng:     {matched}  ({matched/total*100:.1f}%)")
print(f"Not matched:           {total - matched}")

Total ingredient rows: 4003
Matched in CosIng:     2999  (74.9%)
Not matched:           1004


In [0]:
# Save the validated, normalized ingredient bridge table
derm_validated.write.mode("overwrite") \
    .saveAsTable("dermstore.dermstoreschema.derm_ingredients_validated")
print("Saved! derm_ingredients_validated is permanent.")

Saved! derm_ingredients_validated is permanent.


In [0]:
# ============================================================
# TRANSFORMATION — Step 3a: Count CosIng matches per PRODUCT
# ============================================================
# For each product, how many of its ingredients matched CosIng?
# Real skincare products = many matches. Junk/devices = few or none.
# We LOOK at the distribution first, then choose a threshold.

matches_per_product = derm_validated.groupBy("product_id").agg(
    F.count("*").alias("total_ingredients"),
    F.count("inci_name").alias("matched_ingredients")  # count() ignores nulls
)

# Show products with the FEWEST matches first — these are the
# suspects (likely non-skincare). Look at where the cutoff should be.
matches_per_product.orderBy("matched_ingredients").show(30, truncate=False)

+----------+-----------------+-------------------+
|product_id|total_ingredients|matched_ingredients|
+----------+-----------------+-------------------+
|15069728  |1                |0                  |
|12905594  |1                |0                  |
|11289934  |1                |0                  |
|12267722  |9                |0                  |
|13904814  |11               |0                  |
|11689517  |1                |0                  |
|12535727  |9                |0                  |
|11146992  |3                |0                  |
|11443354  |1                |0                  |
|12915728  |1                |0                  |
|14197979  |19               |1                  |
|13888954  |1                |1                  |
|13927692  |2                |1                  |
|15231808  |5                |1                  |
|13882823  |7                |2                  |
|12905677  |13               |2                  |
|12918861  |8                |2

In [0]:
# ============================================================
# TRANSFORMATION — Step 3b: Filter products to skincare (≥2 matches)
# ============================================================
# Using the match counts as evidence: keep products with at least 2
# ingredients recognized by CosIng. This drops junk/devices/haircare
# that have 0-1 matches, while keeping real skincare products.

# (matches_per_product was built in the previous step)
# Keep only products with 2+ matched ingredients
skincare_products = matches_per_product.filter(F.col("matched_ingredients") >= 2)

# See how many products survive vs get dropped
before = matches_per_product.count()
after = skincare_products.count()
print(f"Products before filtering: {before}")
print(f"Products after filtering:  {after}")
print(f"Dropped as non-skincare:   {before - after}")

Products before filtering: 120
Products after filtering:  106
Dropped as non-skincare:   14


In [0]:
# Optional: see which products were DROPPED, to confirm they're really non-skincare
dropped_ids = matches_per_product.filter(F.col("matched_ingredients") < 2)
df_derm.join(dropped_ids, "product_id").select("product_id", "brand", "category").show(20, truncate=70)

+----------+---------------------+----------------------------------------------------------------------+
|product_id|                brand|                                                              category|
+----------+---------------------+----------------------------------------------------------------------+
|  12905594|                  Pmd|                                    Brands / PMD / PMD Clean (1 piece)|
|  13904814|      The Light Salon|              Brands / The Light Salon / The Light Salon Boost LED Bib|
|  12535727| Herbivore Botanicals|Brands / Herbivore Botanicals / Herbivore Botanicals Phoenix Facial...|
|  11289934|           Supersmile|Brands / Supersmile / Supersmile Professional Whitening System - La...|
|  11443354|                  Pmd|                     Brands / PMD / PMD Personal Microderm Man (1 kit)|
|  15231808|             Erborian|Brands / Erborian / Erborian Travel Size CC Cream 14g (Various Shades)|
|  14197979|                Rahua|            

In [0]:
# ============================================================
# TRANSFORMATION — Step 4: Build the concern → ingredient knowledge base
# ============================================================
# This is the "brain" of the recommender: it maps each skin CONCERN
# to the cosmetic ingredients well-established to help with it.
# Built from general, widely-accepted cosmetic skincare knowledge
# (cosmetic guidance, NOT medical treatment claims).
#
# Ingredient names are UPPERCASE to match our cleaned ingredient data.

# Each tuple is (concern, ingredient). One concern maps to several ingredients.
kb_rows = [
    # Acne / breakouts
    ("acne", "SALICYLIC ACID"), ("acne", "NIACINAMIDE"),
    ("acne", "BENZOYL PEROXIDE"), ("acne", "AZELAIC ACID"), ("acne", "RETINOL"),

    # Oily skin
    ("oily skin", "NIACINAMIDE"), ("oily skin", "SALICYLIC ACID"), ("oily skin", "ZINC PCA"),

    # Dryness / dehydration
    ("dryness", "SODIUM HYALURONATE"), ("dryness", "GLYCERIN"),
    ("dryness", "SQUALANE"), ("dryness", "CERAMIDE NP"), ("dryness", "PANTHENOL"),

    # Dark spots / hyperpigmentation
    ("hyperpigmentation", "NIACINAMIDE"), ("hyperpigmentation", "ASCORBIC ACID"),
    ("hyperpigmentation", "AZELAIC ACID"), ("hyperpigmentation", "ALPHA-ARBUTIN"),
    ("hyperpigmentation", "TRANEXAMIC ACID"),

    # Fine lines / aging
    ("aging", "RETINOL"), ("aging", "ASCORBIC ACID"),
    ("aging", "SODIUM HYALURONATE"), ("aging", "NIACINAMIDE"),

    # Redness / sensitivity
    ("redness", "NIACINAMIDE"), ("redness", "AZELAIC ACID"),
    ("redness", "CENTELLA ASIATICA EXTRACT"), ("redness", "ALLANTOIN"), ("redness", "PANTHENOL"),

    # Dullness / uneven texture
    ("dullness", "ASCORBIC ACID"), ("dullness", "GLYCOLIC ACID"),
    ("dullness", "LACTIC ACID"), ("dullness", "NIACINAMIDE"),

    # Enlarged pores
    ("large pores", "NIACINAMIDE"), ("large pores", "SALICYLIC ACID"), ("large pores", "RETINOL"),

    # Sun protection
    ("sun protection", "ZINC OXIDE"), ("sun protection", "TITANIUM DIOXIDE"),

    # Dehydration (barrier)
    ("dehydration", "SODIUM HYALURONATE"), ("dehydration", "GLYCERIN"), ("dehydration", "PANTHENOL"),

    # Blackheads / clogged pores
    ("blackheads", "SALICYLIC ACID"), ("blackheads", "RETINOL"),

    # Rough / bumpy texture
    ("texture", "GLYCOLIC ACID"), ("texture", "LACTIC ACID"), ("texture", "SALICYLIC ACID"),

    # Loss of firmness / elasticity
    ("firmness", "RETINOL"), ("firmness", "ASCORBIC ACID"), ("firmness", "SODIUM HYALURONATE"),

    # Dark circles / under-eye
    ("dark circles", "NIACINAMIDE"), ("dark circles", "ASCORBIC ACID"),
]

# Turn the list into a Spark DataFrame with two columns
kb = spark.createDataFrame(kb_rows, ["concern", "ingredient"])

print("Knowledge base rows:", kb.count())
print("Concerns covered:", kb.select("concern").distinct().count())
kb.orderBy("concern", "ingredient").show(60, truncate=False)

Knowledge base rows: 49
Concerns covered: 14
+-----------------+-------------------------+
|concern          |ingredient               |
+-----------------+-------------------------+
|acne             |AZELAIC ACID             |
|acne             |BENZOYL PEROXIDE         |
|acne             |NIACINAMIDE              |
|acne             |RETINOL                  |
|acne             |SALICYLIC ACID           |
|aging            |ASCORBIC ACID            |
|aging            |NIACINAMIDE              |
|aging            |RETINOL                  |
|aging            |SODIUM HYALURONATE       |
|blackheads       |RETINOL                  |
|blackheads       |SALICYLIC ACID           |
|dark circles     |ASCORBIC ACID            |
|dark circles     |NIACINAMIDE              |
|dehydration      |GLYCERIN                 |
|dehydration      |PANTHENOL                |
|dehydration      |SODIUM HYALURONATE       |
|dryness          |CERAMIDE NP              |
|dryness          |GLYCERIN        

In [0]:
# Save the knowledge base as a permanent, editable table
kb.write.mode("overwrite").saveAsTable("dermstore.dermstoreschema.concern_ingredient_kb")
print("Saved! concern_ingredient_kb ")

Saved! concern_ingredient_kb 


In [0]:
# ============================================================
# TRANSFORMATION — Step 5: Tag products with the concerns they treat
# ============================================================
# Connect product ingredients -> knowledge base -> concerns.
# Result: for each product, which skin concerns does it address?

# 1) Load the validated bridge table (product -> ingredient) we saved earlier
derm_validated = spark.table("dermstore.dermstoreschema.derm_ingredients_validated")

# 2) Keep only the SKINCARE products (the 106 that passed the >=2 filter)
#    skincare_products holds those product_ids from the filtering step.
derm_skincare = derm_validated.join(skincare_products, "product_id")

# 3) Join product ingredients to the knowledge base on the ingredient name.
#    The bridge's normalized ingredient = the KB's ingredient.
#    This is an INNER join: keep only rows where the ingredient IS in the KB
#    (i.e. ingredients that actually map to a concern).
product_concerns = derm_skincare.join(
    kb,
    derm_skincare.ingredient_norm == kb.ingredient,
    how="inner"
).select("product_id", "concern").distinct()   # distinct: one row per product-concern

print("Product-concern tags:", product_concerns.count())
print("\nExample — concerns tagged for a few products:")
product_concerns.orderBy("product_id").show(30, truncate=False)

Product-concern tags: 420

Example — concerns tagged for a few products:
+----------+-----------------+
|product_id|concern          |
+----------+-----------------+
|10360329  |firmness         |
|10360329  |hyperpigmentation|
|10360329  |dullness         |
|10360329  |dehydration      |
|10360329  |dryness          |
|10360329  |dark circles     |
|10360329  |aging            |
|10524496  |dehydration      |
|10524496  |firmness         |
|10524496  |aging            |
|10524496  |dryness          |
|11207519  |aging            |
|11207519  |firmness         |
|11207519  |dryness          |
|11207519  |dehydration      |
|11220951  |dehydration      |
|11220951  |dryness          |
|11264625  |acne             |
|11264625  |oily skin        |
|11264625  |dryness          |
|11264625  |aging            |
|11264625  |redness          |
|11264625  |hyperpigmentation|
|11264625  |dullness         |
|11264625  |dark circles     |
|11264625  |dehydration      |
|11264625  |firmness        

In [0]:
# ============================================================
# TRANSFORMATION — Step 6a: Build the GOLD products table
# ============================================================
# Join the 106 skincare product IDs back to the full Dermstore details
# to get name, brand, price, ingredients — the recommender's catalog.

gold_products = df_derm.join(skincare_products, "product_id").select(
    "product_id",
    "brand",
    "category",
    "price",
    "ingredients_list",
    "skin_type_and_concerns",   # the original (sparse) label, kept for reference
    "rating_value"
)

print("Gold products:", gold_products.count())
gold_products.select("product_id", "brand", "price").show(10, truncate=50)

Gold products: 106
+----------+----------------+-----+
|product_id|           brand|price|
+----------+----------------+-----+
|  12696151|Christophe Robin| 43.0|
|  12902323|            R+co|114.0|
|  15512135|   Colorescience| 59.0|
|  13441436|       Tripollar| 98.0|
|  14645944|        Pca Skin|140.0|
|  10360329|         Alterna| 37.0|
|  12540840|            Neom| 24.0|
|  11486957|  Paula's Choice| 22.0|
|  14869207|        Masktini| 84.0|
|  15061315|    Estée Lauder| 45.0|
+----------+----------------+-----+
only showing top 10 rows


In [0]:
# ============================================================
# TRANSFORMATION — Step 6b: Save BOTH gold tables
# ============================================================
# Gold table 1: the product catalog (106 skincare products)
gold_products.write.mode("overwrite") \
    .saveAsTable("dermstore.dermstoreschema.gold_products")

# Gold table 2: the recommendation lookup (product -> concern, 420 tags)
product_concerns.write.mode("overwrite") \
    .saveAsTable("dermstore.dermstoreschema.gold_product_concerns")

print("Saved both gold tables!")
print(" - gold_products (the catalog)")
print(" - gold_product_concerns (the recommendation lookup)")

Saved both gold tables!
 - gold_products (the catalog)
 - gold_product_concerns (the recommendation lookup)


In [0]:
# ============================================================
# DEMO — Prove it works: recommend products for a concern
# ============================================================
# Pick a concern and find matching products with their details.
user_concern = "acne"   # try changing this to "dryness", "hyperpigmentation", etc.

recommendations = product_concerns.filter(F.col("concern") == user_concern) \
    .join(gold_products, "product_id") \
    .select("product_id", "brand", "price", "concern") \
    .orderBy("price")

print(f"Products recommended for: {user_concern}")
recommendations.show(20, truncate=50)

Products recommended for: acne
+----------+--------------+-----+-------+
|product_id|         brand|price|concern|
+----------+--------------+-----+-------+
|  15327207|           Dhc| 19.0|   acne|
|  11264625|    Illamasqua| 37.0|   acne|
|  12902295|Miriam Quevedo| 45.0|   acne|
|  15418357|         Coola| 48.0|   acne|
|  15081764|  Agent Nateur| 58.0|   acne|
|  14020614|  Perricone Md| 69.0|   acne|
|  11434924|   Dermalogica| 76.0|   acne|
|  14226753|Miriam Quevedo| 80.0|   acne|
|  14869207|      Masktini| 84.0|   acne|
|  15510189|        Eltamd| 86.0|   acne|
|  13175660|Miriam Quevedo| 95.0|   acne|
|  14979647|  Estée Lauder| 95.0|   acne|
|  14897389| Doctor Rogers| 98.0|   acne|
|  13663086|    Strivectin| 99.0|   acne|
|  15449270|Miriam Quevedo|115.0|   acne|
|  14662431|  Sisley Paris|115.0|   acne|
|  11289086|      Pca Skin|118.0|   acne|
|  15508780|   Dermalogica|129.0|   acne|
|  14016854|       Vi Derm|130.0|   acne|
|  14215551|    Skinmedica|133.5|   acne|
+--

In [0]:
# ============================================================
# DEMO — Prove it works: recommend products for TWO concerns
# ============================================================
# Test two different concerns to confirm the recommender works broadly.

for user_concern in ["acne", "hyperpigmentation"]:
    print(f"\n{'='*55}")
    print(f"Products recommended for: {user_concern.upper()}")
    print('='*55)

    recommendations = product_concerns.filter(F.col("concern") == user_concern) \
        .join(gold_products, "product_id") \
        .select("product_id", "brand", "price") \
        .orderBy("price")

    print(f"Found {recommendations.count()} products")
    recommendations.show(10, truncate=40)


Products recommended for: ACNE
Found 22 products
+----------+--------------+-----+
|product_id|         brand|price|
+----------+--------------+-----+
|  15327207|           Dhc| 19.0|
|  11264625|    Illamasqua| 37.0|
|  12902295|Miriam Quevedo| 45.0|
|  15418357|         Coola| 48.0|
|  15081764|  Agent Nateur| 58.0|
|  14020614|  Perricone Md| 69.0|
|  11434924|   Dermalogica| 76.0|
|  14226753|Miriam Quevedo| 80.0|
|  14869207|      Masktini| 84.0|
|  15510189|        Eltamd| 86.0|
+----------+--------------+-----+
only showing top 10 rows

Products recommended for: HYPERPIGMENTATION
Found 23 products
+----------+----------------+-----+
|product_id|           brand|price|
+----------+----------------+-----+
|  15327207|             Dhc| 19.0|
|  10360329|         Alterna| 37.0|
|  11264625|      Illamasqua| 37.0|
|  12696151|Christophe Robin| 43.0|
|  12902295|  Miriam Quevedo| 45.0|
|  15418357|           Coola| 48.0|
|  15081764|    Agent Nateur| 58.0|
|  14020614|    Perricone 

In [0]:
# ============================================================
# DEMO — Recommend products for TWO concerns (with concern column)
# ============================================================
for user_concern in ["acne", "hyperpigmentation"]:
    print(f"\n{'='*55}")
    print(f"Products recommended for: {user_concern.upper()}")
    print('='*55)

    recommendations = product_concerns.filter(F.col("concern") == user_concern) \
        .join(gold_products, "product_id") \
        .select("product_id", "brand", "concern", "price") \
        .orderBy("price")

    print(f"Found {recommendations.count()} products")
    recommendations.show(10, truncate=40)


Products recommended for: ACNE
Found 22 products
+----------+--------------+-------+-----+
|product_id|         brand|concern|price|
+----------+--------------+-------+-----+
|  15327207|           Dhc|   acne| 19.0|
|  11264625|    Illamasqua|   acne| 37.0|
|  12902295|Miriam Quevedo|   acne| 45.0|
|  15418357|         Coola|   acne| 48.0|
|  15081764|  Agent Nateur|   acne| 58.0|
|  14020614|  Perricone Md|   acne| 69.0|
|  11434924|   Dermalogica|   acne| 76.0|
|  14226753|Miriam Quevedo|   acne| 80.0|
|  14869207|      Masktini|   acne| 84.0|
|  15510189|        Eltamd|   acne| 86.0|
+----------+--------------+-------+-----+
only showing top 10 rows

Products recommended for: HYPERPIGMENTATION
Found 23 products
+----------+----------------+-----------------+-----+
|product_id|           brand|          concern|price|
+----------+----------------+-----------------+-----+
|  15327207|             Dhc|hyperpigmentation| 19.0|
|  10360329|         Alterna|hyperpigmentation| 37.0|
|  1

In [0]:
# ============================================================
# DEMO — Products that treat BOTH dullness AND hyperpigmentation
# ============================================================
# We want products tagged with BOTH concerns, not just one.
# Logic: filter to the two concerns, then keep products that
# match BOTH (i.e. have 2 distinct concerns from our pair).

target_concerns = ["dullness", "hyperpigmentation"]

both = product_concerns.filter(F.col("concern").isin(target_concerns)) \
    .groupBy("product_id") \
    .agg(F.countDistinct("concern").alias("num_matched")) \
    .filter(F.col("num_matched") == 2)        # == 2 means it matched BOTH

# Join back to get product details
result = both.join(gold_products, "product_id") \
    .select("product_id", "brand", "price") \
    .orderBy("price")

print(f"Products treating BOTH {target_concerns[0]} AND {target_concerns[1]}:")
print(f"Found {result.count()} products")
result.show(20, truncate=40)

Products treating BOTH dullness AND hyperpigmentation:
Found 23 products
+----------+----------------+-----+
|product_id|           brand|price|
+----------+----------------+-----+
|  15327207|             Dhc| 19.0|
|  11264625|      Illamasqua| 37.0|
|  10360329|         Alterna| 37.0|
|  12696151|Christophe Robin| 43.0|
|  12902295|  Miriam Quevedo| 45.0|
|  15418357|           Coola| 48.0|
|  15081764|    Agent Nateur| 58.0|
|  14020614|    Perricone Md| 69.0|
|  11434924|     Dermalogica| 76.0|
|  14226753|  Miriam Quevedo| 80.0|
|  14869207|        Masktini| 84.0|
|  15510189|          Eltamd| 86.0|
|  13175660|  Miriam Quevedo| 95.0|
|  13663086|      Strivectin| 99.0|
|  11287247|      Dermaquest|102.0|
|  15449270|  Miriam Quevedo|115.0|
|  11289086|        Pca Skin|118.0|
|  14016854|         Vi Derm|130.0|
|  11837067|        Pca Skin|130.0|
|  14215551|      Skinmedica|133.5|
+----------+----------------+-----+
only showing top 20 rows
